In [ ]:
from pathlib import Path
import os
import logging
import pandas as pd
import requests

geoapify_key = os.environ['GEOAPIFY_API_KEY']
candidates = [Path('Weather_Database/WeatherPy_database.csv'), Path('../Weather_Database/WeatherPy_database.csv'), Path('WeatherPy_database.csv')]
file_path = next((path for path in candidates if path.is_file()), None)
if file_path is None:
    raise FileNotFoundError('WeatherPy_database.csv was not found.')
city_data_df = pd.read_csv(file_path)
city_data_df = city_data_df.rename(columns={'Max Temp': 'Max Temperature', 'Current Description': 'Weather Description', 'Lat': 'Latitude', 'Lng': 'Longitude'})
city_data_df.head()


In [ ]:
min_temp = float(input('Minimum temperature (C): '))
max_temp = float(input('Maximum temperature (C): '))
if min_temp > max_temp:
    raise ValueError('Minimum temperature must not exceed maximum temperature.')
preferred_cities_df = city_data_df.loc[city_data_df['Max Temperature'].between(min_temp, max_temp)].copy()
hotel_df = preferred_cities_df.dropna(subset=['Latitude', 'Longitude']).copy()
hotel_df['Hotel Name'] = pd.NA


In [ ]:
radius = 5000
with requests.Session() as client:
    for index, row in hotel_df.iterrows():
        latitude, longitude = row['Latitude'], row['Longitude']
        params = {'categories': 'accommodation.hotel', 'apiKey': geoapify_key, 'limit': 1, 'filter': f'circle:{longitude},{latitude},{radius}', 'bias': f'proximity:{longitude},{latitude}'}
        response = client.get('https://api.geoapify.com/v2/places', params=params, timeout=30)
        if not response.ok:
            raise RuntimeError(f'API request failed with status {response.status_code}.')
        features = response.json().get('features', [])
        if features:
            hotel_df.loc[index, 'Hotel Name'] = features[0].get('properties', {}).get('name', pd.NA)
        logging.info('Completed hotel search for %s', row['City'])
clean_hotel_df = hotel_df.dropna(subset=['Hotel Name']).copy()
clean_hotel_df.to_csv('WeatherPy_vacation.csv', index=False)
clean_hotel_df.head()


In [ ]:
import hvplot.pandas

clean_hotel_df.hvplot.points('Longitude', 'Latitude', geo=True, tiles='OSM', hover_cols=['City', 'Country', 'Hotel Name', 'Max Temperature'], color='Max Temperature', cmap='coolwarm', size=60)
